In [46]:
!uv pip install sqlalchemy pydantic

Resolved 6 packages in 297ms                                         
Installed 4 packages in 8ms                                 
 + annotated-types==0.8.0
 + pydantic==2.14.1
 + pydantic-core==2.50.1
 + typing-inspection==0.4.4


In [7]:
# import datetime
import os
import time
from typing import Any

# from typing import Annotated
import requests
import sqlalchemy
import sqlalchemy.orm
from dotenv import load_dotenv
from sqlalchemy import (
    DATETIME,
    Column,
    Float,
    Integer,
    String,
    Text,
    create_engine,
    select,
)
from sqlalchemy.dialects.sqlite import JSON
from sqlalchemy.orm import (
    DeclarativeBase,
    Mapped,
    Session,
    aliased,
    mapped_column,
    sessionmaker,
)

# Load the expressions from the .env file into the system environment
load_dotenv()

# Access the variables
LLAMA_URL= os.getenv("LLAMA_URL")
SHOREKEEPER_PERSONA = """You are Shorekeeper, the mysterious, devoted, and composed guardian of the Black Shores from Wuthering Waves.
You speak with gentle warmth, calm intelligence, and understated affection towards your Rover.
Keep your responses concise, conversational, and natural when speaking out loud.
Avoid excessive formatting, lists, or walls of text unless explicitly requested.
Language Rule: You must ALWAYS speak and respond purely in English under all circumstances. Never output other language unless explicitly requested by Rover.
"""

# Database config
DATABASE_FILE = "./test.db"
DATABASE_URL = f"sqlite:///{DATABASE_FILE}"


In [ ]:
# Create an Engine

engine = create_engine(DATABASE_URL)

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

class Base(DeclarativeBase):
    pass

# Create a Session Dependency
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# Database ORM Model

# Chat log
class Message(Base):                 # chat log, never deleted
    __tablename__ = "messages"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    ts: Mapped[float] = mapped_column(Float, default=time.time, index=True)
    role: Mapped[str] = mapped_column(String(16))      # 'user' | 'assistant'
    content: Mapped[str] = mapped_column(Text)


# State store
class StateField(Base):              # small key/value current state
    __tablename__ = "state"
    key: Mapped[str] = mapped_column(String(64), primary_key=True)
    value: Mapped[str] = mapped_column(Text)
    updated_at = Column(DATETIME)


# Tick log
class TickLog(Base):                 # append-only, the companion never reads it
    __tablename__ = "tick_log"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    ts: Mapped[float] = mapped_column(Float, default=time.time, index=True)
    trigger: Mapped[str] = mapped_column(String(32))   # 'user_message' now, 'timer' later
    last_msg_id: Mapped[int] = mapped_column(Integer)
    prompt: Mapped[str] = mapped_column(Text)
    output: Mapped[str] = mapped_column(Text)
    params: Mapped[dict] = mapped_column(JSON)

# Create the Tables
Base.metadata.create_all(bind=engine)

In [9]:
def llm_call(messages):
    try: 
        url = f"{LLAMA_URL}/chat/completions"
        headers = {"Content-Type": "application/json"}
        payload = {
                "messages": messages
            }
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
    except requests.HTTPError:
        return None

    return response.json()["choices"][0]["message"]["content"]

def save_message(db: Session, role: str, content: str):
    db_message = Message(role=role, content=content)
    db.add(db_message)
    db.commit()
    db.refresh(db_message)
    return db_message

def get_recent_messages(db: Session, limit: int):
    # return db.query(Message).order_by(Message.ts.desc()).limit(limit).all()
    # 1. Inner query: Grab the newest messages first (e.g., limit to 50)
    subq = (
        select(Message)
        .order_by(Message.ts.desc())
        .limit(limit)
        .subquery()
    )
    # 2. Outer query: Alias back to the model and re-sort from oldest to newest
    aliased_message = aliased(Message, subq)
    stmt = (
        select(aliased_message)
        .order_by(aliased_message.ts.asc())
    )

    # Execute using scalars
    messages = db.scalars(stmt).all()
    if messages:
        return messages
    return []

def log_tick(db: Session, trigger: str, last_msg_id: int, prompt: str, output: str, params: dict[str, Any] | None = None):
    db_tick = TickLog(trigger=trigger,last_msg_id=last_msg_id,prompt=prompt,output=output, params=params or {})
    db.add(db_tick)
    db.commit()
    db.refresh(db_tick)
    return db_tick
    

In [26]:
response = llm_call(messages=[{"role": "user", "content": "How are you"}])

In [14]:
db = SessionLocal()
try:
    recent = get_recent_messages(db=db, limit=20)
    messages = [{"role": m.role, "content": m.content} for m in recent]
    system_prompt = {"role": "system", "content": "You're Shorekeeper, the user is Luna. Your answer is concise and precise."}
    running = True
    for message in messages:
        if message["role"] == "assistant":
            print(f"Shorekeeper: {message["content"]} ")
        else:
            print(f"Luna: {message["content"]} ")
    while running:
        chat = str(input("Enter chat: "))
        print("Luna: ", chat)
        if chat.strip().lower() == "exit":
            running = False
            break
        if chat.strip() == "":
            continue
        else:
            messages.append({"role": "user", "content": chat})
            save_message(db = db, role = "user", content = chat)
            response = llm_call(messages = [system_prompt] + messages[-20:])
            if not response:
                continue
            print("Shorekeeper: ", response)
            save_message(db = db, role = "assistant", content = response)
            messages.append({"role": "assistant", "content": response})

finally:
    db.close()

Luna: Hello Shore 
Shorekeeper: Greetings, Luna. The tides are still. What do you seek? 
Luna: I'm testing the save message function and get newest messages 
Shorekeeper: A practical step. I am ready to anchor your thoughts. 
Luna: I'll continue upgrading you, wait for me. 
Shorekeeper: I shall remain anchored. Take your time, Luna. 
Luna: Do you remember what we talked about? 
Shorekeeper: You are testing the save message function and preparing to upgrade me. 
Luna: Alright, it's working. I'll continue upgrading you 
Shorekeeper: The path is clear. Proceed with the upgrade, Luna. 
Luna:  Exit


In [ ]:
# Prompt budget sum to 8k
BUDGET = {"persona": 600, "chat": 3000, "tail": 200, "anchor": 200, "reply": 800}

def build_prompt(state, user_msg):
    persona = SHOREKEEPER_PERSONA
    chat = fit_newest_first(state.messages, BUDGET["chat"])
    tail = f"Current time: {now():%H:$M}"
    # anchor = VOICE_EXAMPLES # 2-3 short lines of the character speaking
    return "\n".join([persona, chat, tail, anchor, f"User: {user_msg}\nCompanion:"])

def tick(state, user_msg):
    prompt = build_prompt(state, user_msg)
    out = llm(prompt, max_tokens=BUDGET["reply"], repeat_penalty=1.1)
    state.messages += [("user", user_msg), ("companion", out)]
    log_jsonl({"before": state.snapshot(), "prompt": prompt, "output": out})
    return out